# Módulo 4 de Analítica de Datos — Arquitectura Medallion con Delta Lake

En este módulo construiremos una **arquitectura Medallion** (también llamada *multi-hop* o *bronze/silver/gold*) sobre Delta Lake. Es el patrón recomendado por Databricks para organizar un **Lakehouse**.

### ¿Qué es la arquitectura Medallion?
Es un patrón de diseño que organiza los datos en **capas** según su grado de calidad y refinamiento. Cada capa tiene un propósito diferente y suele vivir en su **propio schema** dentro de Unity Catalog:

- **🥉 Bronze (raw / ingesta)**: copia fiel de los datos de origen, **sin transformaciones de negocio**. Sirve como *source of truth* y permite reprocesar las capas superiores si cambian las reglas. Típicamente solo se aplica casteo de tipos, marca de tiempo de ingesta y nombres consistentes.
- **🥈 Silver (cleansed / conformed)**: datos **limpios y validados**: deduplicación, nulos tratados, tipos correctos, joins entre fuentes, columnas derivadas. Sigue siendo a nivel de transacción (granularidad fina), pero apto para análisis.
- **🥇 Gold (curated / business-level)**: datos **agregados y modelados** para casos de uso concretos: KPIs, dashboards, ML features, *star schemas*. Suelen ser tablas más pequeñas y orientadas al consumo.

### ¿Por qué separarlo en capas?
- **Reproducibilidad**: si la lógica de negocio cambia, se reconstruye Silver/Gold sin volver a ingerir.
- **Aislamiento de fallos**: un bug en Gold no contamina Bronze/Silver.
- **Gobernanza**: permisos diferenciados por capa (p. ej., analistas leen Gold, ingenieros tocan Bronze).
- **Trazabilidad**: lineage en Unity Catalog muestra el flujo completo.
- **Rendimiento**: las consultas de BI golpean Gold, que está pre-agregada.

> **Consideración opcional**: en producción, esta arquitectura se suele implementar con **Lakeflow Declarative Pipelines** (antes Delta Live Tables, DLT), donde cada capa se declara y Databricks gestiona dependencias, calidad (`expectations`) y orquestación.

![medallion](./Assets/med_new.png)

## 1. Carga de los CSV en un Volume de Unity Catalog

Como en módulos anteriores, los **Volumes** son la ubicación gobernada para ficheros no tabulares (CSV, JSON, imágenes, modelos, etc.). Aquí volveremos a ingerir los CSV de ventas (2019, 2020 y 2021) que ya usamos en el Módulo 2.

### Recordatorio rápido
- Sintaxis: `CREATE VOLUME [IF NOT EXISTS] <catalog>.<schema>.<nombre>` (sin prefijos usa los actuales).
- `IF NOT EXISTS` hace la sentencia **idempotente**.
- Ruta POSIX accesible: `/Volumes/<catalog>/<schema>/<volume>/...`.

In [ ]:
%sql
-- Creamos el Volume si aún no existe (operación idempotente).
CREATE VOLUME IF NOT EXISTS spark_lab

In [ ]:
import requests

# Resolvemos el catálogo activo dinámicamente para que el notebook sea portable
# entre entornos (dev/test/prod) con distintos catálogos.
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Ruta base del Volume.
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# Ficheros que vamos a descargar (datos de ventas de 2019, 2020 y 2021).
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

# Descargamos cada fichero del repositorio público y lo guardamos en el Volume.
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()  # Error si la respuesta HTTP no es 2xx.

    # 'wb' = write binary (response.content devuelve bytes).
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

## 2. Lectura inicial de los CSV en un DataFrame

Hacemos una lectura preliminar para inspeccionar el contenido. Como **no especificamos esquema**, todas las columnas se leerán como `string` y, al no haber `header=True`, las cabeceras tendrán nombres genéricos (`_c0`, `_c1`, ...).

Este paso **no es la capa Bronze definitiva**: es una exploración rápida antes de decidir el esquema.

In [ ]:
# El glob '*_edited.csv' lee de golpe los tres ficheros (2019, 2020, 2021).
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv')
display(df.limit(100))

## 3. Capa Bronze — Definición del esquema

Para la capa Bronze definimos un **esquema explícito** con `StructType` + `StructField`. En arquitecturas Medallion la convención es:

- En **Bronze** se preservan los datos lo más fieles posible al origen, pero ya con tipos correctos para que las capas siguientes consuman algo predecible.
- Se evita `inferSchema` porque añade una pasada extra y puede equivocarse con tipos ambiguos.

### Tipos de datos clave (recordatorio)
| Tipo Spark | Uso |
|---|---|
| `StringType` | Texto |
| `IntegerType` / `LongType` | Enteros 32 / 64 bits |
| `FloatType` / `DoubleType` | Coma flotante |
| `DecimalType(p,s)` | Decimal exacto (recomendado para dinero) |
| `DateType` / `TimestampType` | Fecha / fecha+hora |

> **Consideración opcional**: en producción es muy habitual añadir en Bronze columnas de **metadatos de ingesta** como `ingestion_timestamp` (`current_timestamp()`), `source_file` (`input_file_name()`) o un `_rescued_data` cuando se usa Auto Loader. Aquí, por simplicidad, las omitimos.

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Esquema explícito para los pedidos de venta. El orden DEBE coincidir con el del CSV.
bronzeSchema = StructType([
    StructField("SalesOrderNumber", StringType()),     # Identificador del pedido
    StructField("SalesOrderLineNumber", IntegerType()), # Línea dentro del pedido
    StructField("OrderDate", DateType()),               # Fecha del pedido
    StructField("CustomerName", StringType()),          # Cliente (nombre completo)
    StructField("Email", StringType()),                 # Email del cliente
    StructField("Item", StringType()),                  # Producto vendido
    StructField("Quantity", IntegerType()),             # Unidades vendidas
    StructField("UnitPrice", FloatType()),              # Precio unitario
    StructField("Tax", FloatType())                     # Impuesto aplicado
])

## 4. Construcción del DataFrame Bronze

Releemos los CSV pasando ahora el esquema. Ya tendremos columnas con nombres significativos y tipos correctos. Este DataFrame será nuestra capa Bronze antes de persistirla.

In [ ]:
# Lectura tipada: ahora cada columna tiene su nombre y tipo según bronzeSchema.
bronze_df = spark.read.load(
    f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv',
    format='csv',
    schema=bronzeSchema
)
display(bronze_df.limit(100))

## 5. Crear el schema (database) `Bronze` en Unity Catalog

En Unity Catalog la jerarquía es **catalog → schema → tabla/volume/función/modelo**. Un *schema* (también llamado *database*) agrupa objetos relacionados. Aquí crearemos un schema por capa: `Bronze`, `Silver` y `Gold`.

### Sintaxis
```sql
CREATE SCHEMA [IF NOT EXISTS] <catalog>.<schema_name>
    [COMMENT '<descripción>']
    [MANAGED LOCATION '<ruta>']
    [WITH DBPROPERTIES (...)]
```

### Importante: sustituye `YOUR_UC_CATALOG_NAME`
El SQL siguiente lleva un placeholder. **Antes de ejecutarlo**, reemplaza `YOUR_UC_CATALOG_NAME` por el nombre real de tu catálogo (lo puedes consultar con `SELECT current_catalog();`).

Si quieres una versión dinámica que use el catálogo actual sin editar a mano, puedes hacerlo desde Python:
```python
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.Bronze")
```

In [ ]:
%sql
-- Sustituye YOUR_UC_CATALOG_NAME por tu catálogo real.
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Bronze;

## 6. Persistir la capa Bronze como tabla Delta

Vamos a guardar la capa Bronze de **dos formas distintas**, lo que ilustra la diferencia entre tabla **path-based** y tabla **registrada en el catálogo**:

1. **Path-based** (en el Volume): `df.write.format("delta").save(path)`. La tabla vive en una carpeta concreta (`_delta_log/` + Parquet). No aparece en Unity Catalog hasta que la registres.
2. **Managed table en Unity Catalog**: `df.write.format("delta").saveAsTable("<schema>.<tabla>")`. Unity Catalog gestiona la ubicación física y se accede por nombre lógico desde SQL/BI.

### Modos de escritura
- `mode('overwrite')` reemplaza el contenido de la tabla (útil para reprocesos completos).
- `mode('append')` añade datos (incremental).
- `mode('errorifexists')` (por defecto) falla si ya existe.
- `mode('ignore')` no hace nada si ya existe.

### Opciones útiles
- `.option("mergeSchema", "true")` para añadir columnas nuevas en un append.
- `.option("overwriteSchema", "true")` para reemplazar el esquema en un overwrite.
- `.partitionBy("col")` para particionar (recuerda: liquid clustering suele ser preferible hoy).

> **Aviso**: el comentario original del notebook decía "Storing in DBFS". En realidad estamos escribiendo en un **Unity Catalog Volume**, no en DBFS. Es un detalle importante porque Volumes son la opción recomendada hoy y soportan clústeres Shared/serverless donde DBFS está restringido.

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# 1) Persistencia path-based en el Volume de Unity Catalog.
bronzeDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/bronze_sales_orders'
bronze_df.write.format('delta').mode('overwrite').save(bronzeDeltaTablePath)

# 2) Registro como managed table en el schema 'Bronze' del catálogo activo.
#    Acceso posterior: SELECT * FROM Bronze.bronze_sales_orders;
bronze_df.write.format('delta').saveAsTable('Bronze.bronze_sales_orders')

## 7. Capa Silver — Limpieza y transformación

La capa **Silver** parte de Bronze y aplica calidad de datos:

- Deduplicación (`dropDuplicates`).
- Tratamiento de nulos (`dropna`, `fillna`, `na.replace`).
- Casteo y normalización de tipos.
- Cálculo de columnas derivadas que sean útiles para múltiples casos de uso.
- Joins entre fuentes para unificar dimensiones (cliente, producto, etc.).

El objetivo es que cualquier consumidor de Silver pueda confiar en que los datos están **limpios y conformados**, sin tener que repetir limpieza.

### 7.1 Leer Bronze como punto de partida de Silver

En lugar de releer los CSV originales, leemos la **tabla Delta Bronze** ya persistida. Esto desacopla las capas: si mañana cambian los CSV, Bronze cambia, pero Silver sigue partiendo de la misma fuente confiable.

### `spark.read.format("delta").table(...)` vs `spark.table(...)`
Ambas formas funcionan para leer una tabla del catálogo. La forma corta es:
```python
silver_df = spark.table("Bronze.bronze_sales_orders")
```

In [ ]:
# Leemos la tabla Bronze por su nombre lógico en el catálogo.
silver_df = spark.read.format("delta").table("Bronze.bronze_sales_orders")
silver_df.show(10)

### 7.2 Limpieza del DataFrame Silver

Aplicamos las transformaciones que convierten Bronze en Silver:

1. **`dropDuplicates()`**: elimina filas idénticas en todas las columnas. Si solo queremos deduplicar por algunas (p. ej., `SalesOrderNumber` + `SalesOrderLineNumber`), pasamos la lista: `dropDuplicates(["SalesOrderNumber", "SalesOrderLineNumber"])`.
2. **Recalcular `Tax`** como el 8 % del `UnitPrice`. Esto normaliza posibles inconsistencias del origen.
3. **`cast("float")`**: garantiza el tipo final de la columna (la multiplicación por un literal puede devolver `Double`).

### Sobre la inmutabilidad y el lazy evaluation
Los DataFrames son **inmutables**: cada `withColumn`/`dropDuplicates` devuelve un DataFrame nuevo. Las transformaciones no se ejecutan aquí; solo se ejecutarán cuando llamemos a una **acción** (como el `write` siguiente).

In [ ]:
from pyspark.sql.functions import col

# 1) Deduplicación exacta en todas las columnas.
silver_df = silver_df.dropDuplicates()

# 2) Recalculamos Tax como el 8% del precio unitario para normalizar el dato.
silver_df = silver_df.withColumn('Tax', col('UnitPrice') * 0.08)

# 3) Aseguramos el tipo Float (la multiplicación por un literal puede dar Double).
silver_df = silver_df.withColumn('Tax', col('Tax').cast("float"))

### 7.3 Crear el schema `Silver`

Igual que con Bronze, creamos un schema dedicado para la capa Silver. Recuerda sustituir `YOUR_UC_CATALOG_NAME` por tu catálogo real.

> **Consideración opcional**: en producción se suele aplicar **gobernanza diferenciada por capa**. Por ejemplo, los analistas pueden tener `SELECT` solo en `Gold`, mientras que `Bronze`/`Silver` están restringidas al equipo de ingeniería:
> ```sql
> GRANT USAGE ON SCHEMA <catalog>.Gold TO `analistas`;
> GRANT SELECT ON SCHEMA <catalog>.Gold TO `analistas`;
> ```

In [ ]:
%sql
-- Sustituye YOUR_UC_CATALOG_NAME por tu catálogo real.
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Silver;

### 7.4 Persistir Silver como tabla Delta

Igual que con Bronze, guardamos Silver de dos formas: path-based (en el Volume) y como managed table en el catálogo (`Silver.silver_sales_orders`).

> **Patrón habitual**: en producción esta escritura se haría con `MERGE INTO` para hacer upserts incrementales (especialmente cuando se aplica **CDC**, *Change Data Capture*, desde Bronze).

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# 1) Persistencia path-based en el Volume.
silverDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/silver_sales_orders'
silver_df.write.format('delta').mode('overwrite').save(silverDeltaTablePath)

# 2) Registro como managed table en el schema 'Silver'.
silver_df.write.format('delta').saveAsTable('Silver.silver_sales_orders')

## 8. Capa Gold — Agregaciones para consumo de negocio

La capa **Gold** sirve datos listos para análisis o consumo final. Puede contener:

- KPIs agregados (ventas anuales, margen mensual, churn rate...).
- Tablas de hechos y dimensiones (modelado dimensional / *star schema*).
- *Feature stores* para Machine Learning.
- Vistas materializadas para dashboards.

Es habitual tener **varias tablas Gold por dominio** (ventas, marketing, finanzas...) construidas a partir de la misma Silver.

### 8.1 Leer Silver como punto de partida de Gold

Igual que antes, partimos de la tabla Silver registrada en el catálogo, no de los CSV. Esto refuerza la separación de capas y aprovecha la calidad ya aplicada en Silver.

In [ ]:
# Leemos Silver. gold_df es un alias inicial: lo transformaremos a un agregado.
gold_df = spark.read.format("delta").table("Silver.silver_sales_orders")
gold_df.show(10)

### 8.2 Agregación de ventas anuales

Calculamos el número de pedidos por año, encadenando transformaciones idiomáticas en PySpark:

- `year("OrderDate")` extrae el año de la fecha.
- `.alias("Year")` renombra la columna calculada (equivale a `AS` en SQL).
- `.groupBy("Year").count()` agrupa por año y cuenta.
- `.orderBy("Year")` ordena ascendentemente.

Equivalente en SQL:
```sql
SELECT year(OrderDate) AS Year, COUNT(*) AS cnt
FROM Silver.silver_sales_orders
GROUP BY year(OrderDate)
ORDER BY Year;
```

> **Visualización opcional**: tras ejecutar `display(yearlySales)`, en Databricks puedes pulsar `+` sobre la salida para añadir un gráfico (barras, líneas...) sin escribir más código.

In [ ]:
# Agregación: nº de pedidos por año, ordenado cronológicamente.
yearlySales = (
    gold_df.select(year("OrderDate").alias("Year"))
           .groupBy("Year")
           .count()
           .orderBy("Year")
)
display(yearlySales)

### 8.3 Crear el schema `Gold` en Unity Catalog

Schema dedicado para la capa Gold. Recuerda sustituir `YOUR_UC_CATALOG_NAME`.

In [ ]:
%sql
-- Sustituye YOUR_UC_CATALOG_NAME por tu catálogo real.
CREATE SCHEMA IF NOT EXISTS YOUR_UC_CATALOG_NAME.Gold;

### 8.4 Persistir Gold como tabla Delta

Guardamos el agregado anual tanto path-based (en el Volume) como managed table en el schema `Gold` del catálogo. La tabla Gold es la que consumirán dashboards, herramientas de BI o servicios downstream.

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# 1) Persistencia path-based en el Volume.
goldDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/gold_sales_orders'
yearlySales.write.format('delta').mode('overwrite').save(goldDeltaTablePath)

# 2) Registro como managed table en el schema 'Gold'.
yearlySales.write.format('delta').saveAsTable('Gold.gold_sales_orders')

## Resumen del módulo

Hemos implementado un pipeline Medallion completo sobre Delta Lake:

1. **Ingesta** de CSV en un Volume de Unity Catalog.
2. **Bronze**: lectura tipada con esquema explícito y persistencia como Delta (path-based + managed table en el schema `Bronze`).
3. **Silver**: lectura desde Bronze, deduplicación y normalización de `Tax`, persistencia en el schema `Silver`.
4. **Gold**: lectura desde Silver y agregación de ventas anuales, persistencia en el schema `Gold`.

### Buenas prácticas a tener en cuenta
- **Idempotencia**: usa `CREATE ... IF NOT EXISTS` y modos de escritura conscientes (`overwrite` para reprocesos completos, `append` o `MERGE INTO` para incremental).
- **Schemas por capa**: `Bronze`/`Silver`/`Gold` permiten gobernanza y permisos diferenciados.
- **Esquemas explícitos**: evitan sorpresas y aceleran la lectura.
- **Trazabilidad**: la pestaña *Lineage* de Unity Catalog mostrará el flujo Bronze → Silver → Gold automáticamente.
- **Calidad**: en producción, añade *expectations* (con DLT/Lakeflow) o validaciones explícitas para detectar datos malos antes de propagarlos.

### Próximos pasos sugeridos
- Reescribir el pipeline como **Lakeflow Declarative Pipelines (DLT)** con `@dlt.table` y `@dlt.expect_or_drop` por capa.
- Sustituir `mode('overwrite')` por **`MERGE INTO`** para upserts incrementales (idempotente y barato).
- Activar **Change Data Feed** (`delta.enableChangeDataFeed = true`) en Bronze para alimentar Silver de forma incremental.
- Aplicar **`OPTIMIZE` + `ZORDER BY`** o **liquid clustering** en Silver/Gold para acelerar consultas frecuentes.
- Definir **expectations** de calidad (`expect`, `expect_or_drop`, `expect_or_fail`) y monitorización con dashboards.